In [1]:
import polars as pl
import gspread
from google.oauth2.service_account import Credentials
from sqlalchemy import create_engine,text
scope = ["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]

creds = Credentials.from_service_account_file("credentials.json", scopes=scope)
client = gspread.authorize(creds)



### Processing Outstanding Sheet

In [7]:
outstanding_sheet = client.open("Income Expense 2082 (6 months)").worksheet('outstanding')

outstanding = pl.DataFrame(outstanding_sheet.get_all_records())
outstanding = outstanding.with_columns(pl.lit('Outstanding').alias("Category"))
outstanding.columns = ["agent_name","party_name","zone","net_due","30days","60days","90days","more_than_90days","category"]
outstanding = outstanding.with_columns(
    pl.col("zone").str.strip_chars().alias("zone")
)

outstanding

agent_name,party_name,zone,net_due,30days,60days,90days,more_than_90days,category
str,str,str,f64,f64,f64,f64,f64,str
"""Jeevan Ghimire""","""Alam Devi Enterprises""","""Zone 5""",941375.16,624750.45,316624.71,0.0,0.0,"""Outstanding"""
"""Jeevan Ghimire""","""Bhagwan Madira Tatha Kirana Pa…","""Zone 5""",154165.23,0.0,0.0,154165.23,0.0,"""Outstanding"""
"""Jeevan Ghimire""","""Bhagwan Suppliers""","""Zone 5""",307560.24,0.0,0.0,307560.24,0.0,"""Outstanding"""
"""Jeevan Ghimire""","""Bhawaneshwor Enterprises""","""Zone 5""",1.7671e7,2.4817e6,5.6129e6,5.5466e6,4.0302e6,"""Outstanding"""
"""Jeevan Ghimire""","""Daya Chandra Suppliers""","""Zone 5""",626530.24,0.0,534510.11,0.0,92020.13,"""Outstanding"""
…,…,…,…,…,…,…,…,…
"""Raju Maharjan""","""North Liquor Station""","""Zone 3""",23500.16,0.0,0.0,0.0,23500.16,"""Outstanding"""
"""Raju Maharjan""","""Rijal Distributors Pvt. Ltd.""","""Zone 3""",349860.25,0.0,187425.13,124950.09,37485.03,"""Outstanding"""
"""Raju Maharjan""","""S K Suppliers""","""Zone 3""",2.5798e6,0.0,1328175.5,499800.36,751835.16,"""Outstanding"""


In [3]:
out_sm_sheet = client.open("Income Expense 2082 (6 months)").worksheet('outstanding sm')
out_sm = pl.DataFrame(out_sm_sheet.get_all_records())
out_sm
 

Zone,Area,Party Details,Closing Amt
str,str,str,f64
"""Zone 3""","""Bhakundebesi""","""Jay Shree Chamunda Suppliers""",6562.96
"""Zone 3 ""","""Dhading""","""B S Traders""",1.7831e6
"""Zone 3""","""Dhading""","""Gajuri Madira Suppliers""",174081.89
"""Zone 3""","""Dhading""","""Galchhi Enterprises""",612218.47
"""Zone 3""","""Dhading""","""S K Suppliers""",19006.12
…,…,…,…
"""Zone 6""","""Zone-6, Kanchanpur""","""R M Store""",233799.55
"""Zone 6""","""Zone-6, Nepalgunj""","""Mata Kalanki Traders""",1.5344e6
"""Zone 6""","""Zone-6, Pyuthan""","""Ashok Stores""",80383.04


In [17]:
out_sm = out_sm.with_columns(
    pl.col("Zone").str.strip_chars().alias("Zone")
)

mapper  = {
    'Zone 1' : "Kiran Adhikari",
    'Zone 2' : "Kiran Adhikari",
    "Zone 2B" : "Madhu Subedi",
    "Zone 3"  : "Raju Maharjan",
    "Zone 5" : "Jeevan Ghimire",
    "Zone 6" : "Jeevan Ghimire"
}
out_sm = out_sm.with_columns(
    pl.col("Zone").replace(mapper, default="Other").alias("Agent")
)

out_sm.columns = ['zone','area','party_name','closing_amt','agent']
out_sm

C:\Users\ASUS\AppData\Local\Temp\ipykernel_25136\2944641951.py:14: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("Zone").replace(mapper, default="Other").alias("Agent")


zone,area,party_name,closing_amt,agent
str,str,str,f64,str
"""Zone 3""","""Bhakundebesi""","""Jay Shree Chamunda Suppliers""",6562.96,"""Raju Maharjan"""
"""Zone 3""","""Dhading""","""B S Traders""",1.7831e6,"""Raju Maharjan"""
"""Zone 3""","""Dhading""","""Gajuri Madira Suppliers""",174081.89,"""Raju Maharjan"""
"""Zone 3""","""Dhading""","""Galchhi Enterprises""",612218.47,"""Raju Maharjan"""
"""Zone 3""","""Dhading""","""S K Suppliers""",19006.12,"""Raju Maharjan"""
…,…,…,…,…
"""Zone 6""","""Zone-6, Kanchanpur""","""R M Store""",233799.55,"""Jeevan Ghimire"""
"""Zone 6""","""Zone-6, Nepalgunj""","""Mata Kalanki Traders""",1.5344e6,"""Jeevan Ghimire"""
"""Zone 6""","""Zone-6, Pyuthan""","""Ashok Stores""",80383.04,"""Jeevan Ghimire"""


In [16]:
collection_sheet = client.open("Income Expense 2082 (6 months)").worksheet('collection')
collect = pl.DataFrame(collection_sheet.get_all_records())

collect = collect.with_columns(
    [
        pl.col('Agent ').str.strip_chars().alias('Agent '),
        pl.col('Zone').str.strip_chars().alias('Zone')
    ]
)
collect.columns = ['agent','party_name','date_','zone','cash_bank','receipt']

collect

agent,party_name,date_,zone,cash_bank,receipt
str,str,str,str,str,i64
"""Jeevan Ghimire""","""Daya Chandra Suppliers""","""5/20/2082""","""Zone 5""","""Nepal Investment Mega Bank Ltd""",200000
"""Jeevan Ghimire""","""Daya Chandra Suppliers""","""6/23/2082""","""Zone 5""","""Nepal Investment Mega Bank Ltd""",200000
"""Jeevan Ghimire""","""Daya Chandra Suppliers""","""8/1/2082""","""Zone 5""","""Nepal Investment Mega Bank Ltd""",150000
"""Jeevan Ghimire""","""Daya Chandra Suppliers""","""9/28/2082""","""Zone 5""","""Nepal Investment Mega Bank Ltd""",150000
"""Jeevan Ghimire""","""Alam Devi Enterprises""","""7/30/2082""","""Zone 5""","""Nepal Investment Mega Bank Ltd""",150000
…,…,…,…,…,…
"""Raju Maharjan""","""Team Trade Pvt.Ltd""","""8/25/2082""","""Zone 3""","""Global IME Bank Ltd""",270000
"""Raju Maharjan""","""Team Trade Pvt.Ltd""","""9/13/2082""","""Zone 3""","""Global IME Bank Ltd""",200000
"""Raju Maharjan""","""Team Trade Pvt.Ltd""","""9/20/2082""","""Zone 3""","""Global IME Bank Ltd""",150000


In [18]:
scheme_sm_sheet = client.open("Income Expense 2082 (6 months)").worksheet('scheme sm')
scheme_sm = pl.DataFrame(scheme_sm_sheet.get_all_records())